In [1]:
!pip install -q gradio
!wget -q https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
!unzip -q -o ml-latest-small.zip
print("Setup complete")

Setup complete


In [2]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
movies = pd.read_csv("ml-latest-small/movies.csv")    # movieId, title, genres
ratings = pd.read_csv("ml-latest-small/ratings.csv")  # userId, movieId, rating, timestamp

In [4]:
print(movies.shape, ratings.shape)
print(f"Users: {ratings.userId.nunique()}, Movies rated: {ratings.movieId.nunique()}")
movies.head()

(9742, 3) (100836, 4)
Users: 610, Movies rated: 9724


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [8]:
#CONTENT-BASED FILTERING (movie genres -> TF-IDF -> cosine similarity)
movies["genres_clean"] = movies["genres"].str.replace("|", " ", regex=False)

tfidf = TfidfVectorizer(token_pattern=r"[^ ]+")
genre_matrix = tfidf.fit_transform(movies["genres_clean"])
content_sim = cosine_similarity(genre_matrix)

title_to_idx = pd.Series(movies.index, index=movies["title"])


def find_title(query):
    """Helper: search titles by substring, e.g. find_title('toy story')"""
    return movies[movies["title"].str.contains(query, case=False, regex=False)][["movieId", "title", "genres"]]


def recommend_by_genre(title, n=10):
    """Recommend movies with genres similar to the given movie (exact title incl. year)."""
    if title not in title_to_idx:
        print("Title not found. Try find_title('part of title').")
        return None
    idx = title_to_idx[title]
    scores = list(enumerate(content_sim[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    scores = [s for s in scores if s[0] != idx][:n]
    result = movies.iloc[[i for i, _ in scores]][["title", "genres"]].copy()
    result["similarity"] = [round(s, 3) for _, s in scores]
    return result.reset_index(drop=True)


print(find_title("toy story"))
recommend_by_genre("Toy Story (1995)", n=10)

      movieId               title  \
0           1    Toy Story (1995)   
2355     3114  Toy Story 2 (1999)   
7355    78499  Toy Story 3 (2010)   

                                                genres  
0          Adventure|Animation|Children|Comedy|Fantasy  
2355       Adventure|Animation|Children|Comedy|Fantasy  
7355  Adventure|Animation|Children|Comedy|Fantasy|IMAX  


,title,genres,similarity
0,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy,1.0
1,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,1.0
2,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.0
3,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.0
4,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,1.0
5,"Wild, The (2006)",Adventure|Animation|Children|Comedy|Fantasy,1.0
6,Shrek the Third (2007),Adventure|Animation|Children|Comedy|Fantasy,1.0
7,"Tale of Despereaux, The (2008)",Adventure|Animation|Children|Comedy|Fantasy,1.0
8,Asterix and the Vikings (Astérix et les Viking...,Adventure|Animation|Children|Comedy|Fantasy,1.0
9,Turbo (2013),Adventure|Animation|Children|Comedy|Fantasy,1.0


In [9]:
#COLLABORATIVE FILTERING (item-based, cosine similarity on mean-centred ratings)
MIN_RATINGS = 20
counts = ratings.groupby("movieId").size()
popular_ids = counts[counts >= MIN_RATINGS].index
r = ratings[ratings["movieId"].isin(popular_ids)]

# User x Movie matrix (NaN = not rated)
user_item = r.pivot_table(index="userId", columns="movieId", values="rating")

# Mean-centre each user's ratings so harsh/generous raters are comparable
user_means = user_item.mean(axis=1)
centered = user_item.sub(user_means, axis=0).fillna(0)

# Item-item similarity
item_sim = cosine_similarity(centered.values.T)
item_sim_df = pd.DataFrame(item_sim, index=user_item.columns, columns=user_item.columns)

movie_titles = movies.set_index("movieId")["title"]
movie_genres = movies.set_index("movieId")["genres"]


def recommend_for_user(user_id, n=10):
    """Predict ratings for movies the user hasn't seen and return the top n."""
    if user_id not in user_item.index:
        print("Unknown user id.")
        return None
    u_vec = centered.loc[user_id].values
    rated_mask = (u_vec != 0).astype(float)

    numer = item_sim @ u_vec
    denom = np.abs(item_sim) @ rated_mask
    denom[denom == 0] = np.nan
    pred = user_means[user_id] + numer / denom

    pred = pd.Series(pred, index=user_item.columns)
    already_seen = user_item.loc[user_id].dropna().index
    pred = pred.drop(already_seen).dropna().clip(0.5, 5.0)

    top = pred.sort_values(ascending=False).head(n)
    return pd.DataFrame({
        "title": movie_titles.loc[top.index].values,
        "genres": movie_genres.loc[top.index].values,
        "predicted_rating": top.round(2).values,
    })


def recommend_similar_movies(title, n=10):
    """'People who liked this also liked...' based on rating patterns."""
    row = movies[movies["title"] == title]
    if row.empty or row.iloc[0]["movieId"] not in item_sim_df.index:
        print("Title not found or has too few ratings.")
        return None
    mid = row.iloc[0]["movieId"]
    sims = item_sim_df[mid].drop(mid).sort_values(ascending=False).head(n)
    return pd.DataFrame({
        "title": movie_titles.loc[sims.index].values,
        "genres": movie_genres.loc[sims.index].values,
        "similarity": sims.round(3).values,
    })


print("Top picks for user 1:")
display(recommend_for_user(1, n=10))

print("Movies similar to 'Toy Story (1995)' by rating patterns:")
display(recommend_similar_movies("Toy Story (1995)", n=10))

Top picks for user 1:


,title,genres,predicted_rating
0,"Producers, The (1968)",Comedy,4.79
1,"Great Escape, The (1963)",Action|Adventure|Drama|War,4.79
2,Superbad (2007),Comedy,4.77
3,Garden State (2004),Comedy|Drama|Romance,4.77
4,Wallace & Gromit: A Close Shave (1995),Animation|Children|Comedy,4.76
5,"Crow, The (1994)",Action|Crime|Fantasy|Thriller,4.76
6,"Departed, The (2006)",Crime|Drama|Thriller,4.75
7,Cool Hand Luke (1967),Drama,4.75
8,Eternal Sunshine of the Spotless Mind (2004),Drama|Romance|Sci-Fi,4.75
9,American Gangster (2007),Crime|Drama|Thriller,4.75


Movies similar to 'Toy Story (1995)' by rating patterns:


,title,genres,similarity
0,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,0.351
1,Aladdin (1992),Adventure|Animation|Children|Comedy|Musical,0.282
2,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,0.279
3,Wallace & Gromit: The Wrong Trousers (1993),Animation|Children|Comedy|Crime,0.264
4,"Incredibles, The (2004)",Action|Adventure|Animation|Children|Comedy,0.224
5,Finding Nemo (2003),Adventure|Animation|Children|Comedy,0.220
6,Blazing Saddles (1974),Comedy|Western,0.211
7,Back to the Future (1985),Adventure|Comedy|Sci-Fi,0.211
8,Babe (1995),Children|Drama,0.200
9,"Lion King, The (1994)",Adventure|Animation|Children|Drama|Musical|IMAX,0.187


In [10]:
def user_favourites(user_id, n=10):
    liked = ratings[ratings.userId == user_id].sort_values("rating", ascending=False).head(n)
    return liked.merge(movies, on="movieId")[["title", "genres", "rating"]]

user_favourites(1)

,title,genres,rating
0,Seven (a.k.a. Se7en) (1995),Mystery|Thriller,5.0
1,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,5.0
2,Bottle Rocket (1996),Adventure|Comedy|Crime|Romance,5.0
3,Dumb & Dumber (Dumb and Dumber) (1994),Adventure|Comedy,5.0
4,Billy Madison (1995),Comedy,5.0
5,Desperado (1995),Action|Romance|Western,5.0
6,Canadian Bacon (1995),Comedy|War,5.0
7,Rob Roy (1995),Action|Drama|Romance|War,5.0
8,Pinocchio (1940),Animation|Children|Fantasy|Musical,5.0
9,Tombstone (1993),Action|Drama|Western,5.0


In [11]:
from sklearn.model_selection import train_test_split

train, test = train_test_split(r, test_size=0.2, random_state=42)

tr_matrix = train.pivot_table(index="userId", columns="movieId", values="rating")
tr_matrix = tr_matrix.reindex(index=user_item.index, columns=user_item.columns)
tr_means = tr_matrix.mean(axis=1).fillna(r["rating"].mean())
tr_centered = tr_matrix.sub(tr_means, axis=0).fillna(0)

S = cosine_similarity(tr_centered.values.T)
num = tr_centered.values @ S
den = (tr_centered.values != 0).astype(float) @ np.abs(S)
den[den == 0] = np.nan
pred_matrix = tr_means.values[:, None] + num / den
pred_df = pd.DataFrame(pred_matrix, index=tr_matrix.index, columns=tr_matrix.columns)

preds, actuals = [], []
for row in test.itertuples():
    p = pred_df.at[row.userId, row.movieId]
    if not np.isnan(p):
        preds.append(np.clip(p, 0.5, 5.0))
        actuals.append(row.rating)

rmse = np.sqrt(np.mean((np.array(preds) - np.array(actuals)) ** 2))
baseline = np.sqrt(np.mean((train["rating"].mean() - np.array(actuals)) ** 2))
print(f"Item-based CF RMSE : {rmse:.3f}")
print(f"Global-mean baseline: {baseline:.3f}")
print(f"Evaluated on {len(preds)} of {len(test)} test ratings")

Item-based CF RMSE : 0.837
Global-mean baseline: 1.008
Evaluated on 13570 of 13580 test ratings


In [12]:
def hybrid_recommend(user_id, n=10, alpha=0.7):
    """
    alpha = weight for collaborative score, (1 - alpha) for content score.
    Content score = avg genre similarity to the user's highly rated movies (rating >= 4).
    """
    cf = recommend_for_user(user_id, n=len(user_item.columns))
    if cf is None:
        return None
    cf = cf.merge(movies[["title", "movieId"]], on="title")

    liked = ratings[(ratings.userId == user_id) & (ratings.rating >= 4)]["movieId"]
    liked_idx = movies.index[movies["movieId"].isin(liked)]
    if len(liked_idx) == 0:
        return cf.head(n)[["title", "genres", "predicted_rating"]]

    content_scores = content_sim[:, liked_idx].mean(axis=1)
    content_series = pd.Series(content_scores, index=movies["movieId"])

    cf["cf_norm"] = (cf["predicted_rating"] - 0.5) / 4.5   # scale to 0..1
    cf["content"] = cf["movieId"].map(content_series)
    cf["hybrid_score"] = (alpha * cf["cf_norm"] + (1 - alpha) * cf["content"]).round(3)
    return cf.sort_values("hybrid_score", ascending=False).head(n)[
        ["title", "genres", "predicted_rating", "hybrid_score"]
    ].reset_index(drop=True)


hybrid_recommend(1, n=10)

,title,genres,predicted_rating,hybrid_score
0,"Great Escape, The (1963)",Action|Adventure|Drama|War,4.79,0.752
1,City of God (Cidade de Deus) (2002),Action|Adventure|Crime|Drama|Thriller,4.66,0.750
2,Laputa: Castle in the Sky (Tenkû no shiro Rapy...,Action|Adventure|Animation|Children|Fantasy|Sc...,4.72,0.750
3,"Lord of the Rings: The Return of the King, The...",Action|Adventure|Drama|Fantasy,4.69,0.748
4,Children of Men (2006),Action|Adventure|Drama|Sci-Fi|Thriller,4.67,0.747
5,"Crow, The (1994)",Action|Crime|Fantasy|Thriller,4.76,0.745
6,Three Kings (1999),Action|Adventure|Comedy|Drama|War,4.67,0.745
7,Inside Out (2015),Adventure|Animation|Children|Comedy|Drama|Fantasy,4.69,0.743
8,"Boondock Saints, The (2000)",Action|Crime|Drama|Thriller,4.74,0.743
9,Seven Samurai (Shichinin no samurai) (1954),Action|Adventure|Drama,4.67,0.743


In [15]:
import gradio as gr

all_titles = sorted(movies["title"].tolist())
rated_titles = sorted(movie_titles.loc[item_sim_df.index].tolist())   # titles with enough ratings
user_ids = [int(u) for u in user_item.index]


def _check_user(user_id):
    uid = int(user_id)
    if uid not in user_item.index:
        raise gr.Error(f"User {uid} not found. Valid IDs: {user_ids[0]} to {user_ids[-1]}.")
    return uid


def ui_genre(title, n):
    res = recommend_by_genre(title, int(n))
    if res is None:
        raise gr.Error("Movie not found.")
    return res


def ui_similar(title, n):
    res = recommend_similar_movies(title, int(n))
    if res is None:
        raise gr.Error("Not enough ratings for this movie. Pick another.")
    return res


def ui_user(user_id, n):
    uid = _check_user(user_id)
    return user_favourites(uid, 10), recommend_for_user(uid, int(n))


def ui_hybrid(user_id, n, alpha):
    uid = _check_user(user_id)
    return hybrid_recommend(uid, int(n), float(alpha))


with gr.Blocks(title="Movie Recommender") as demo:
    gr.Markdown("# 🎬 Movie Recommendation System\nContent-based, collaborative and hybrid filtering on MovieLens.")

    with gr.Tab("By genre (content-based)"):
        t1 = gr.Dropdown(all_titles, value="Toy Story (1995)", label="Pick a movie", filterable=True)
        n1 = gr.Slider(3, 20, value=10, step=1, label="Number of recommendations")
        out1 = gr.Dataframe(headers=["title", "genres", "similarity"], interactive=False, wrap=True,label="Movies with similar genres")
        gr.Button("Recommend", variant="primary").click(ui_genre, [t1, n1], out1)

    with gr.Tab("Similar movies (collaborative)"):
        t2 = gr.Dropdown(rated_titles, value="Toy Story (1995)", label="Pick a movie", filterable=True)
        n2 = gr.Slider(3, 20, value=10, step=1, label="Number of recommendations")
        out2 = gr.Dataframe(headers=["title", "genres", "similarity"], interactive=False, wrap=True,label="People who liked this also liked")
        gr.Button("Recommend", variant="primary").click(ui_similar, [t2, n2], out2)

    with gr.Tab("For a user (collaborative)"):
        u3 = gr.Number(value=1, precision=0, label=f"User ID ({user_ids[0]} to {user_ids[-1]})")
        n3 = gr.Slider(3, 20, value=10, step=1, label="Number of recommendations")
        fav3 = gr.Dataframe(headers=["title", "genres", "rating"], interactive=False, wrap=True,label="Movies this user rated highest")
        out3 = gr.Dataframe(headers=["title", "genres", "predicted_rating"], interactive=False, wrap=True,label="Recommended for this user")
        gr.Button("Recommend", variant="primary").click(ui_user, [u3, n3], [fav3, out3])

    with gr.Tab("Hybrid"):
        u4 = gr.Number(value=1, precision=0, label=f"User ID ({user_ids[0]} to {user_ids[-1]})")
        n4 = gr.Slider(3, 20, value=10, step=1, label="Number of recommendations")
        a4 = gr.Slider(0, 1, value=0.7, step=0.05, label="Alpha (1 = all collaborative, 0 = all genre-based)")
        out4 = gr.Dataframe(headers=["title", "genres", "predicted_rating", "hybrid_score"],interactive=False, wrap=True, label="Hybrid recommendations")
        gr.Button("Recommend", variant="primary").click(ui_hybrid, [u4, n4, a4], out4)

    demo.load(ui_genre, [t1, n1], out1)
    demo.load(ui_similar, [t2, n2], out2)
    demo.load(ui_user, [u3, n3], [fav3, out3])
    demo.load(ui_hybrid, [u4, n4, a4], out4)

    demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://65d2dea7609d5bfad7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
